# 综合案例

## 1、RAG结合大模型的小例子

第1种情况：不使用RAG

In [1]:
from langchain_openai import ChatOpenAI
import os
import dotenv
dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 创建大模型实例
llm = ChatOpenAI(model="gpt-4o-mini")

# 调用
response = llm.invoke("北京有什么著名的建筑？")
print(response.content)

北京有许多著名的建筑，以下是一些代表性的建筑：

1. **故宫**：又称紫禁城，是中国明清两代的皇家宫殿，现为故宫博物院，拥有丰富的历史和文化遗产。

2. **天安门广场**：世界上最大的城市广场之一，周围有天安门、人民大会堂、国家博物馆等重要建筑。

3. **天坛**：明清两代的皇帝祭天的地方，以其独特的建筑风格和精美的园林设计闻名。

4. **颐和园**：清朝的皇家园林，以其美丽的湖泊、山丘和古典建筑而著称，是中国古代园林艺术的杰出代表。

5. **长城**：虽然长城是一个延绵的防御工程，但北京地区的八达岭、慕田峪等段落非常著名，吸引了大量游客。

6. **鸟巢（国家体育场）**：为2008年北京奥运会而建的现代建筑，以其独特的外观和结构著称。

7. **水立方（国家游泳中心）**：也为2008年奥运会而建，以其水泡状的外观和先进的技术而闻名。

8. **北京大学和清华大学**：这两所高校不仅在学术上享有盛誉，其校园内的建筑和自然环境也颇具特色。

9. **798艺术区**：一个充满现代艺术气息的区域，许多改建的工业建筑成为艺术展览和文化活动的场所。

这些建筑不仅代表了北京的历史文化，也展示了中国的建筑艺术和现代化进程。


第2种情况：使用RAG

In [2]:
# 1. 导入所有需要的包
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores import FAISS
import os
import dotenv

dotenv.load_dotenv()

# 2. 创建自定义提示词模板
prompt_template = """请使用以下提供的文本内容来回答问题。仅使用提供的文本信息，如果文本中没有相关信息，请回答"抱歉，提供的文本中没有这个信息"。

文本内容：
{context}

问题：{question}

回答：
"
"""

prompt = PromptTemplate.from_template(prompt_template)

# 3. 初始化模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

# 4. 加载文档
loader = TextLoader("./asset/load/10-test_doc.txt", encoding='utf-8')
documents = loader.load()

# 5. 分割文档
text_splitter = CharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=0,
    length_function=len
)
texts = text_splitter.split_documents(documents)

print(f"文档个数:{len(texts)}")

# 6. 创建向量存储
vectorstore = FAISS.from_documents(
    documents=texts,
    embedding=embeddings
)

# 7.获取检索器
retriever = vectorstore.as_retriever()

docs = retriever.invoke("北京有什么著名的建筑？")

# 8. 创建Runnable链
chain = prompt | llm

# 9. 提问
result = chain.invoke(input={"question":"北京有什么著名的建筑？","context":docs})
print("\n回答:", result.content)

文档个数:1

回答: 北京有以下著名的建筑：

1. 故宫 - 明清两代的皇家宫殿，世界上现存规模最大、保存最完整的木质结构古建筑群之一。
2. 天安门 - 北京的标志性建筑之一，天安门广场是世界上最大的城市广场。
3. 颐和园 - 清朝时期的皇家园林，融合了江南园林的设计风格。
4. 天坛 - 明清两代皇帝祭天、祈谷的场所，具有深厚的文化内涵。
5. 长城（八达岭段） - 最著名的北京段，被誉为"世界第八大奇迹"。
6. 国家体育场（鸟巢） - 2008年奥运会主体育场，以独特的钢结构设计著称。
7. 中央电视台总部大楼 - 现代北京最具争议和识别度的建筑之一。
8. 国家大剧院 - 因其蛋壳造型被称为"巨蛋"，是世界最大的穹顶建筑之一。
9. 北京大兴国际机场 - 超大型国际航空枢纽，被誉为"新世界七大奇迹"之一。
10. 鼓楼和钟楼 - 古代中国的计时中心，展现了古代的计时智慧。


## 2、项目：智能对话助手

1、提示词模板：ChatPromptTemplate、PromptTemplate等

2、大语言模型的调用：ChatOpenAI() 、 OpenAI()

3、(熟悉)输出解析器：JSONOutputParser、StrOutputParser

4、Chain的使用：LLMChain、SequentialChain; | 、create_sql_query_chain

5、关于模型调用前，给模型提供上下文信息：memory：ChatMessageHistory、ConversationBufferMemory

6、工具的创建；工具的使用，给Agent

7、Agent的使用：Agent ---> AgentExecutor   ---> LangGraph

传统的方式：initialize_agent(llm = llm,agent=AgentType.XXX,tools=tools)

通用的方式：create_xxx_agent(llm=llm,tools=tools,prompt=prompt)
             AgentExecutor(agent=agent,toos=tools)

8、数据的本地存储（使用向量数据库）：RAG
   具体的流程：XxxLoader --> list[Document] --> XxxSplitter ---> list[Document] -->
             OpenAIEmbeddings() ---> FAISS/Chroma  ---> Retriever


## 步骤1：定义工具


In [3]:
import os
from langchain_community.tools.tavily_search import TavilySearchResults

# 定义 AVILY_KEY 密钥
import dotenv
dotenv.load_dotenv()
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY") 
# 查询 Tavily 搜索 API
search = TavilySearchResults(max_results=1)
# 执行查询
res = search.invoke("今天上海天气怎么样")
print(res)

[{'url': 'https://weather.cma.cn/web/weather/58367.html', 'content': '|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 14.5℃ | 17.3℃ | 18℃ | 16.1℃ | 14.2℃ | 13.9℃ | 13.5℃ | 13.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 |\n| 风速 | 7.9m/s | 7.9m/s | 7.8m/s | 7.4m/s | 7.3m/s | 7.8m/s | 7.9m/s | 7.8m/s |\n| 风向 | 西北风 | 东北风 | 东北风 | 东北风 | 西北风 | 无持续风向 | 东北风 | 东北风 |\n| 气压 | 1029hPa | 1029.1hPa | 1027.4hPa | 1027.2hPa | 1027.2hPa | 1026.8hPa | 1025.5hPa | 1024.7hPa | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 19.5℃ | 20.2℃ | 22.6℃ | 21.6℃ | 18.6℃ | 18.3℃ | 18℃ | 16.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 |\n| 风速 | 7.9m/s | 7.9m/s | 7.8m/s | 7.7m/s | 7.8m/s | 7.9m/s | 7.9m/s | 7.9m/s |\n| 风向 | 东北风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 | 东南风 |\n| 气压 | 1017.6hPa | 1016.2hPa | 10

## 步骤2：Retriever

In [4]:
os.environ["USER_AGENT"] = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0 Safari/537.36"
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import os
import dotenv
dotenv.load_dotenv()

# 1. 提供一个大模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

embedding_model = OpenAIEmbeddings()

# 2.加载HTML内容为一个文档对象


loader = WebBaseLoader("https://zh.wikipedia.org/wiki/%E7%8C%AB")
docs = loader.load()
#print(docs)

# 3.分割文档
documents = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
).split_documents(docs)

# 4.向量化 得到向量数据库对象
vector = FAISS.from_documents(documents, embedding_model)

# 5.创建检索器
retriever = vector.as_retriever()

# 测试检索结果
#print(retriever.invoke("猫的特征")[0])

USER_AGENT environment variable not set, consider setting it to identify your requests.


## 步骤3：创建工具、工具集


In [5]:
from langchain.tools.retriever import create_retriever_tool

# 创建一个工具来检索文档
retriever_tool = create_retriever_tool(
    retriever=retriever,
    name="wiki_search",
    description="搜索维基百科",
)

tools = [search, retriever_tool]

## 步骤4：语言模型调用工具

In [6]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 获取大模型
model = ChatOpenAI(model="gpt-4o-mini")

# 模型绑定工具
# model_with_tools = model.bind_tools(tools)
#
# # 根据输入自动调用工具
# response = model_with_tools.invoke([HumanMessage(content="今天上海天气怎么样")])
# print(f"ContentString: {response.content}")
# print(f"ToolCalls: {response.tool_calls}")


## 步骤5：创建Agent程序(使用通用方式)

In [7]:
from langchain import hub
prompt = hub.pull("hwchase17/openai-functions-agent")

prompt.messages

D:\BaiduSyncdisk\pytorch\AI_agent\langchain_env\Lib\site-packages\langsmith\client.py:241: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [8]:
from langchain.agents import create_tool_calling_agent
from langchain.agents import AgentExecutor
# 创建Agent对象
agent = create_tool_calling_agent(model, tools, prompt)

# 创建AgentExecutor对象
agent_executor = AgentExecutor(agent=agent, tools=tools,verbose=True)

## 步骤6：运行Agent


In [9]:
print(agent_executor.invoke({"input": "猫的特征"}))



> Entering new AgentExecutor chain...

Invoking: `wiki_search` with `{'query': '猫的特征'}`


感官[编辑]
貓的感官適於狩獵。在哺乳類動物中，貓的聽覺、視覺、嗅覺、味覺、觸覺極敏銳。

視覺[编辑]
貓的瞳孔能縮得如線般狹小
貓的瞳孔變化
貓在晝間視覺縱不及人類，夜視能力與追蹤視覺上之活動物件卻相當出色，夜視能力是人類的六倍，雖然綜合色彩計算整體視覺系數則僅及人類的十分之一。貓的眼睛具有微光觀察能力，即使只有微弱月光都可分辨物件，有關光線入貓眼後可放大40至50倍處理，令貓具有夜間活動的能力，即使在黑暗的地下室貓咪依然能活動自如。強光下，貓會將瞳孔縮得如線般狹小，以減少對視網膜的傷害，但視野會因而縮窄。由於貓眼具備高幀與高分辨率視覺，故此電視機極微細之動靜對貓而言亦成逐格跳躍之畫面。[51]
另外，不只是光線會影響貓的瞳孔，感情也會。一般而言貓放鬆的時候瞳孔會縮小，而緊張時會放大。
貓的視網膜背面有一層藍綠色如熒光一般的薄膜（Tapetum Lucidum），可增加在暗處的視力。閃光中，貓眼能呈現各式各樣顔色。如同多數食肉動物，貓眼長在臉上朝正前方，賦予其遼闊的視野，單一貓眼視野為100度，雙眼視野為285度，加上頸旋靈活，總計視野比人類雙眼僅100度靈活得多。不過，貓僅能聚焦其前30厘米至3米之物件，相對人類而言屬大近視。
貓對三原色的辨識力很差，相對人類而言，貓有黃藍色盲且紅綠色辨成灰黃色，與狗相似。關於貓的夜視能力，生物學家發現牛磺酸對貓的視力起了很大作用，貓本身不能合成牛磺酸，必須由外攝取。缺乏牛磺酸，會使視力及夜視能力變差，影響夜間活動。據生物學家稱，貓經常捕鼠，是因老鼠體內含牛磺酸。[52][53]
紫外線可以通過貓的晶状体。
當四周光線微弱，貓會用感覺毛來改善行動力與感知能力。感覺毛主要分布於鼻子兩側、下巴、雙眼上方、兩頰也有數根。感覺毛可感受非常微弱的空氣波動，視野不清時也能協助辨識阻礙位置。鬍鬚尖端與雙耳連成一線，恰是身體能通過障礙的最小範圍，故貓可在黑夜中快速判斷地形能否通過。
貓有第三眼瞼，當貓眼睑張開時，眨眼時第三眼瞼會從旁稍微遮蓋眼睛。若貓生病，或是睡眠，笑著，此眼皮會縮回一部分。若貓長時間嶄露第三眼瞼，表示它的健康有問題。

^ 生物學特性和解剖生理

In [10]:
print(agent_executor.invoke({"input": "今天上海天气怎么样"}))



> Entering new AgentExecutor chain...

Invoking: `tavily_search_results_json` with `{'query': '今天上海天气'}`


[{'url': 'https://weather.cma.cn/web/weather/58367.html', 'content': '|  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 14.5℃ | 17.3℃ | 18℃ | 16.1℃ | 14.2℃ | 13.9℃ | 13.5℃ | 13.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 |\n| 风速 | 7.9m/s | 7.9m/s | 7.8m/s | 7.4m/s | 7.3m/s | 7.8m/s | 7.9m/s | 7.8m/s |\n| 风向 | 西北风 | 东北风 | 东北风 | 东北风 | 西北风 | 无持续风向 | 东北风 | 东北风 |\n| 气压 | 1029hPa | 1029.1hPa | 1027.4hPa | 1027.2hPa | 1027.2hPa | 1026.8hPa | 1025.5hPa | 1024.7hPa | [...] |  |  |  |  |  |  |  |  |  |\n ---  ---  ---  --- \n| 时间 | 08:00 | 11:00 | 14:00 | 17:00 | 20:00 | 23:00 | 02:00 | 05:00 |\n| 天气 |\n| 气温 | 19.5℃ | 20.2℃ | 22.6℃ | 21.6℃ | 18.6℃ | 18.3℃ | 18℃ | 16.5℃ |\n| 降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 | 无降水 |\n| 风速 | 7.9m/s | 7.9m/s | 7.8m/s | 7.7m/s | 7.8m/s | 7.9m/s |

## 步骤7：添加记忆

In [ ]:
# 导入聊天历史记录相关模块
from langchain_community.chat_message_histories import ChatMessageHistory

from langchain_core.chat_history import BaseChatMessageHistory

from langchain_core.runnables.history import RunnableWithMessageHistory

# 用于存储不同会话的聊天历史
store = {}

# 调取指定session_id对应的memory
def get_session_history(session_id: str) -> BaseChatMessageHistory:
    # 如果当前session_id不存在，则新建一个ChatMessageHistory实例
    if session_id not in store:
        store[session_id] = ChatMessageHistory()

    return store[session_id]

# 将agent_executor包装为支持历史记录的runnable
agent_with_chat_history = RunnableWithMessageHistory(
    runnable=agent_executor,# agent_executor，使用的agent对象
    get_session_history=get_session_history,# 获取指定session_id的memory
    input_messages_key="input",# 输入消息的key
    history_messages_key="chat_history",# 历史消息的key
)

# 调用带历史记录的agent，传入用户输入和会话ID
response = agent_with_chat_history.invoke(
    {"input": "Hi，我的名字是Cyber"},
    config={"configurable": {"session_id": "123"}},#指定会话ID
)

# 打印agent返回的响应
print(response)



> Entering new AgentExecutor chain...
你好，Cyber！很高兴认识你。有什么我可以帮助你的吗？

> Finished chain.
{'input': 'Hi，我的名字是Cyber', 'chat_history': [], 'output': '你好，Cyber！很高兴认识你。有什么我可以帮助你的吗？'}


In [12]:
response = agent_with_chat_history.invoke(
    {"input": "我的名字叫什么？"},
    config={"configurable": {"session_id": "123"}},#从存储的memory中获取
)

print(response)



> Entering new AgentExecutor chain...
你的名字是Cyber。有什么特别想讨论或了解的内容吗？

> Finished chain.
{'input': '我的名字叫什么？', 'chat_history': [HumanMessage(content='Hi，我的名字是Cyber', additional_kwargs={}, response_metadata={}), AIMessage(content='你好，Cyber！很高兴认识你。有什么我可以帮助你的吗？', additional_kwargs={}, response_metadata={})], 'output': '你的名字是Cyber。有什么特别想讨论或了解的内容吗？'}


In [13]:
response = agent_with_chat_history.invoke(
    {"input": "我的名字叫什么？"},
    config={"configurable": {"session_id": "345"}}, #因为这是345会话，所以无法获取123的 memory
)

print(response)



> Entering new AgentExecutor chain...
抱歉，我无法知道您的名字。如果您愿意，可以告诉我您的名字，我会记住的。

> Finished chain.
{'input': '我的名字叫什么？', 'chat_history': [], 'output': '抱歉，我无法知道您的名字。如果您愿意，可以告诉我您的名字，我会记住的。'}
